# Introduction

The notebook for calculating intrument functions.
For more details see signal_processing_pipeline.ipynb.

# Libraries


In [1]:
import mascope_lib.runtime as lib_runtime
lib_runtime.init()

import numpy as np
from datetime import datetime
from mascope_lib.file_func import get_instrument_type, get_sum_signal, load_array
from mascope_lib.peak import detect_peaks, gen_peak
from mascope_lib.instrument_functions import process_peak_shapes, fit_fwhm, get_resolution_function
from mascope_lib.inst_func_viz import vizualize
import mascope_api
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from IPython.utils.capture import capture_output

# Spectrum overview

In [ ]:
filename = "KLTOF1_2024.09.26_17h17m07s_+"
instrument_type = get_instrument_type(filename)

# Extract sum spectrum
sum_signal = get_sum_signal(filename)
mz = sum_signal.mz.values
spec = sum_signal.values

# Plot overview
fig = go.Figure([go.Scatter(x=mz, y=spec, name="Raw counts")])
fig.update_layout(
    height=300,
    width=800,
    margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    xaxis_title="M/z [Th]",
    yaxis_title="Counts [a.u.]",
)
fig.show()

# Calculate peak shape


In [ ]:
dmz = 0.5
r_squared_threshold = 0.95
n_peaks = 100

# Get x-domain, normalized peak shapes and associated peak positions and FWHMs
p_x, p_ys, p_mzs, p_fwhms = process_peak_shapes(
    mz, spec, instrument_type, dmz, r_squared_threshold, n_peaks
)

# Create peak shape traces
peak_norm_traces = [
    go.Scatter(x=p_x, y=p_y, name=f"peak {i}")
    for i, p_y in enumerate(p_ys)
    ]

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)

# Check if p_median is empty
if p_median.all() == np.nan:
    raise Exception(
        """p_median is empty
        Probably fitting error threshold is too strict"""
    )

# Add to the figure
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)
# Plot the output
fig = go.FigureWidget(
    peak_norm_traces,
    {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"},
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)


def toggle_trace_visibility(trace, points, selector):
    """Updates the median peak shape based on shapes present in the figure"""
    global p_median
    if not points.point_inds:
        return

    trace.visible = "legendonly"
    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]
    p_median = np.median(np.array(selected_traces), axis=0)

    fig.update_layout(
        title=f"Median peak shape of {len(selected_traces)} peaks",
        height=300,
        width=800,
        margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    )
    fig.update_traces(
        {"y": p_median},
        selector=lambda trace: trace.name == "median",
    )


print("Click on distorted peaks to remove them from peak shape estimation.")

[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

Save median peak shape


In [ ]:
# Save peak
peak_shape = {"x": p_x, "y": p_median}
p_ys = np.array(p_ys)
# Calculate residuals for normalized peaks vs median peak shape
norm_residuals = p_ys - p_median
## calculate R-squared vs mz
r_sq_norm = 1 - (norm_residuals**2).sum(axis=1) / ((p_ys - p_ys.mean()) ** 2).sum(
    axis=1
)
# Plot error vs mz
err_fig = plt.figure(figsize=(7, 1.5))
plt.scatter(p_mzs, r_sq_norm)
plt.xlabel("mz")
plt.ylabel(r"R$^2$");

# Calculate resolution function


In [5]:
# Convert values to numpy arrays
p_mzs = np.array(p_mzs)
p_fwhms = np.array(p_fwhms)

# Get fitted FWHM vs m/z pairs
p_fwhms_fit = fit_fwhm(instrument_type, p_mzs, p_fwhms)

# Number of std to filter out outliers in FWHM fit
ndev = 1

resolution_function = get_resolution_function(instrument_type, p_mzs, p_fwhms, ndev)

# Fit peaks

In [27]:
filename = "KLTOF1_2024.09.26_17h17m07s_+"

# Extract sum spectrum
sum_signal = get_sum_signal(filename)

In [36]:
# Fit the peaks for the current resolution function and peak shape
# Rich output may hang the system, that's why we catch it and save in cap
with capture_output() as cap:
    sample_file_data = await detect_peaks(
        filename,
        (peak_shape, resolution_function),
        0.9,
        u_list=p_mzs,  # for testing
        max_n_peaks=10,
        if_exists="replace",
        dmz=0.5,
        instrument_type=instrument_type,
    )
# Get fitted peak positions and heights
fit_heis = (
    load_array(filename, "peak_heights")
    .dropna(dim="mz")
    .sum(dim="time")["peak_heights"]
)
fit_poss = fit_heis.mz.values
fit_heis = fit_heis.values
# NOTE all the outputs can be seen with cap.show()
# the list of outputs accessed with cap.outputs 

# Visualization

In [7]:
subtitles = ("FWHM", "Chosen peak", "Resolution function")


def update_chosen_peak(trace, points, selector):
    """Update chosen peak in resolution function visualization"""
    if points.xs:
        # Clean peak shapes
        fig_widget.layout.shapes = []
        # Clean annotations
        fig_widget.layout.annotations = [
            i for i in fig_widget.layout.annotations if i.text in subtitles
        ]
        chosen_peak_val = points.xs[0]
        # Calculate chosen peak window width
        dmz = 3 * chosen_peak_val / resolution_function(chosen_peak_val)
        # Mask area around chosen_peak_val
        chosen_peak_mask = (chosen_peak_val - dmz < mz) & (mz < chosen_peak_val + dmz)
        # Filter spectra window to plot
        mz_window_x = mz[chosen_peak_mask]
        mz_window_y = spec[chosen_peak_mask]
        # Init fitted signal
        fit_y = np.zeros_like(mz_window_x)

        # Fitted peak mask
        fit_peak_mask = (chosen_peak_val - dmz < fit_poss) & (
            fit_poss < chosen_peak_val + dmz
        )
        fit_poss_filt = fit_poss[fit_peak_mask]
        fit_heis_filt = fit_heis[fit_peak_mask]
        # Add fitted peaks to fit_y
        for i, mz_val in enumerate(fit_poss_filt):
            fit_y += gen_peak(
                mz_window_x,
                mz_val,
                fit_heis_filt[i],
                pres=resolution_function(mz_val),
                ps=peak_shape,
            )
            fig_widget.add_shape(
                type="line",
                x0=mz_val,
                x1=mz_val,
                y0=0,
                y1=fit_heis_filt[i],
                line=dict(width=2),
                xref="x2",
                yref="y2",
            )
            fig_widget.add_annotation(
                x=mz_val,
                y=fit_heis_filt[i],
                text=f"{mz_val:.3f}",
                showarrow=False,
                xref="x2",
                yref="y2",
                yshift=10,
            )
        # Residuals
        residuals = mz_window_y - fit_y

        fig_widget.update_traces(
            {"x": mz_window_x, "y": mz_window_y},
            selector=lambda trace: trace.name == "Chosen peak",
        )
        fig_widget.update_traces(
            {"x": mz_window_x, "y": fit_y},
            selector=lambda trace: trace.name == "Fitted signal",
        )
        fig_widget.update_traces(
            {"x": mz_window_x, "y": residuals},
            selector=lambda trace: trace.name == "Residuals",
        )

In [ ]:
# Get spectrum arrays for visualization
spec = sum_signal.values
mz = sum_signal.mz.values

fig = vizualize(p_mzs, p_fwhms, p_fwhms_fit, ndev, resolution_function)

fig_widget = go.FigureWidget(fig)

fig_widget.data[4].on_click(update_chosen_peak)
fig_widget.data[5].on_click(update_chosen_peak)

fig_widget

### Troubleshooting:

* There are too many or too little data points used in the calculation -> increase or decrease `n_peaks` value.
* Too many / too little resolution data points are excluded -> edit `ndev` value. If it's increased, more values will go gray.

# Save instrument functions


In [ ]:
# Get the current UTC date and time
datetime_utc = datetime.now()
# Convert the datetime object to a string in UTC format
datetime_utc_str = datetime_utc.isoformat()

# Convert peak shape to lists for json
peak_shape_lists = {
    "x": peak_shape["x"].tolist(),
    "y": peak_shape["y"].tolist()
}
# Get resolution function coefficients from partial
res_fun_coefs = resolution_function.keywords
if instrument_type =="tof":
    res_fun_list = [res_fun_coefs["a"]]
else:
    res_fun_list = [res_fun_coefs["a"], res_fun_coefs["b"]]

mascope_api.create_instrument_function(
    mascope_url="http://localhost:8080/",
    instrument="KORBI2",
    datetime_utc=datetime_utc_str,
    peakshape=peak_shape_lists,
    resolution_function=res_fun_list,
)